In [ ]:
# wavelength interpolation

import dLux as dl
import jax.numpy as np
import jax
import matplotlib.pyplot as plt
import dLux.utils as dlu
from dLux.sources import Source
from jax import Array
from dLux.sources import BaseSource
from jax import vmap
import jax.random as jr
from dLux import spectra
from matplotlib.style import use
use('default')
jax.config.update("jax_enable_x64", True)
plt.rcParams['image.origin'] = 'lower'

from jax.scipy.special import gammaln, xlogy

def poisson_loglike_gamma(k, mu):
    # analytic continuation using gammaln(k+1)
    k = np.asarray(k, dtype=float)
    mu = np.asarray(mu, dtype=float)
    valid = (k >= 0) & np.isfinite(k) & (mu > 0) & np.isfinite(mu)
    logp = xlogy(k, mu) - mu - gammaln(k + 1.0)
    return np.where(valid, logp, -np.inf)

# define the likelihood function:
def poiss_loglike(pytree, data) -> float:
    """
    Poissonian log likelihood of the pytree given the data. Assumes the pytree
    has a .model() function.

    Parameters
    ----------
    pytree : Base
        Pytree with a .model() function.
    data : Array
        Data to compare the model to.

    Returns
    -------
    log_likelihood : Array
        Log likelihood of the pytree given the data.
    """
    return poisson_loglike_gamma(pytree.model(), data).sum()

def normal_logpdf(x, mu, sigma):
    z = (x-mu)/sigma
    return -0.5 * (np.log(2*np.pi) + 2*np.log(sigma) + z**2)


# JATIS style: 8 pt fonts, 5.25 in width, 0.5 pt spines and ticks (see jatis_style.py)
import jatis_style as js
js.apply()

fig_width = js.FIG_WIDTH


In [ ]:
eps = 1e-15
class CombinedSpectrum(dl.Spectrum):
    wavelengths: Array
    filt_weights: Array
    basis_weights: Array
    weights: Array
    flux: float

    def __init__(self, wavels, filt_weights, basis_weights):
        self.wavelengths = np.asarray(wavels, dtype=float)
        self.filt_weights = np.asarray(filt_weights, dtype=float)
        self.basis_weights = np.asarray(basis_weights, dtype=float)

        spec_w = self.spec_weights()
        weights = self.filt_weights * spec_w
        self.weights = weights / weights.sum()
        self.flux = np.sum(spec_w)

    def spec_weights(self):
        raise NotImplementedError

    def normalise(self):
        return self


class CombinedBasisSpectrum(CombinedSpectrum):
    basis_vects: Array

    def __init__(self, wavels, filt_weights, basis_weights, basis):
        self.basis_vects = np.asarray(basis, dtype=float)
        super().__init__(wavels, filt_weights, basis_weights)

    def spec_weights(self):
        return np.maximum(np.sum(self.basis_vects * self.basis_weights, axis=1), eps)
        # return 10 ** (np.sum(self.basis_vects * self.basis_weights, axis=1))

In [ ]:
np.logspace(3,5,5,base=2)

In [ ]:
from zodiax.diffops import hessian

# make a grid from this:
# fovs = 8*np.logspace(0,2,5,base=2)# np.array([8,16,32])
fovs = np.array([4, 8, 16, 32, 64])
flux = 1e6 # 1 million photons... that's a nice amount
read_vars = np.array([0, 2, 4, 8, 16, 32, 64]) # do it per Nyquist resolution element

nconstrained_grid = np.zeros((len(fovs), len(read_vars)), dtype=int)

# construct the aperture once
wf_npixels = 1024
diameter = 1.0
wf_oversample = 5
coords = dlu.pixel_coords(wf_npixels * wf_oversample, diameter)

primary = dlu.circle(coords, diameter / 2)
aperture = dlu.combine(np.array([primary]), wf_oversample)

aperture_layer = dl.TransmissiveLayer(aperture, True)

layers = [
    (
        'aperture',
        aperture_layer
    )
]

def calculate_nconstrained_modes(fov, flux, read_var, threshold=0.1):
    n_wavels_data = 101

    # Important: psf_npixels must be an integer
    psf_npixels = int(np.round(2 * fov))


    psf_pixel_scale = dlu.rad2arcsec((1.0e-6) / 2)  # nyquist
    oversample = 2

    sigma_r = np.sqrt( read_var**2 / (oversample**2) ) # combine in quadrature, there's a quarter of the variance.
 
    optics = dl.AngularOpticalSystem(
        wf_npixels=wf_npixels,
        diameter=diameter,
        layers=layers,
        psf_npixels=psf_npixels,
        psf_pixel_scale=psf_pixel_scale,
        oversample=oversample
    )

    wlmin_data = 1.0e-6
    wlmax_data = 2.0e-6

    wavelengths_data = np.linspace(wlmin_data, wlmax_data, n_wavels_data)

    nbasis = n_wavels_data
    bandpass = np.ones(n_wavels_data)

    spectrum_bins = CombinedBasisSpectrum(
        wavelengths_data,
        bandpass,
        np.ones(nbasis),
        np.eye(nbasis)
    )

    source_data = dl.PointSource(
        spectrum=spectrum_bins
    )

    model_data = dl.Telescope(optics, source_data)

    def model_psf(spectrum, instrument):
        old_spec = instrument.get("source.spectrum")

        new_spec = CombinedBasisSpectrum(
            wavels=old_spec.wavelengths,
            filt_weights=old_spec.filt_weights,
            basis_weights=spectrum,
            basis=old_spec.basis_vects,
        )

        telescope = instrument.set("source.spectrum", new_spec)
        telescope = telescope.set("source.flux", telescope.get("source.spectrum").flux)
        return telescope.model()

    injected_spectrum = np.ones(n_wavels_data) / n_wavels_data * flux

    img_full = model_psf(injected_spectrum, model_data)

    def loss_fn(params, model, data):
        spectrum = params['spectrum']
        img = model_psf(spectrum, model)
        loss = -normal_logpdf(data, img, np.sqrt(data+sigma_r**2)).sum()
        return loss

    params = {
        "spectrum": injected_spectrum
    }

    f = lambda params: loss_fn(params, model_data, img_full)

    H, unflatten = hessian(f, params)

    H = np.asarray(H)
    H = 0.5 * (H + H.T)

    eigvals, eigvecs = np.linalg.eigh(H)

    idx = np.argsort(eigvals)[::-1]

    eigvals = eigvals[idx]
    eigvecs = eigvecs[:, idx]

    # Optional sign convention
    for i in range(eigvecs.shape[0]):
        eigvecs = eigvecs.at[:, i].mul(np.sign(np.mean(eigvecs[:1, i])))

    a = eigvecs.T @ injected_spectrum

    relstd = 1 / (np.abs(a[0]) * np.sqrt(eigvals))

    # safer than np.argmax(relstd > threshold)
    # because np.argmax returns 0 if nothing is True
    bad = np.where(relstd > threshold)[0]

    if len(bad) == 0:
        nconstrained_modes = len(relstd)
    else:
        nconstrained_modes = bad[0]

    return nconstrained_modes

In [ ]:
for i, fov in enumerate(fovs):
    for j, read_var in enumerate(read_vars):
        print(f"fov={fov:.2f}, read_var={read_var:.2f}")

        nconstrained_grid = nconstrained_grid.at[i, j].set(calculate_nconstrained_modes(
            fov=fov,
            flux=flux,
            read_var=read_var,
            threshold=0.1
        ))

        print(nconstrained_grid[i, j])

In [ ]:
for i in range(len(fovs)):
    plt.plot(read_vars, nconstrained_grid[i, :])

In [ ]:
# fixed angular fov, downsample factor
# fixed n_pix, downsample factor (same plot)

# make a grid from this:
# fov = 32
fovs = np.array([4,8,16,32,64])#8*np.logspace(0,2,3,base=2)# np.array([8,16,32])
flux = 1e6 # 1 million photons... that's a nice amount
sample_facs = np.array([0.5, 1, 2, 4, 8]) # 
nconstrained_sample = np.zeros((len(fovs), len(sample_facs)), dtype=int)

# construct the aperture once
wf_npixels = 1024
diameter = 1.0
wf_oversample = 5
coords = dlu.pixel_coords(wf_npixels * wf_oversample, diameter)

primary = dlu.circle(coords, diameter / 2)
aperture = dlu.combine(np.array([primary]), wf_oversample)

aperture_layer = dl.TransmissiveLayer(aperture, True)

layers = [
    (
        'aperture',
        aperture_layer
    )
]

def calculate_nconstrained_modes_sample(fov, flux, sample_fac, threshold=0.1):
    n_wavels_data = 101

    oversample = 2

    sample_fac_os = int(round(oversample * sample_fac))

    if not np.isclose(sample_fac_os, oversample * sample_fac):
        raise ValueError(
            "oversample * sample_fac must be an integer for Downsample."
        )

    # Nominal Nyquist grid spanning fov lambda/D
    psf_npixels = int(np.ceil(2 * fov))

    # dLux's nominal, pre-oversampling pixel scale
    lambda_over_D = 1.0e-6
    psf_pixel_scale = dlu.rad2arcsec(
        fov * lambda_over_D / psf_npixels
    )

    # Detector acts on the internally oversampled image
    internal_npixels = psf_npixels * oversample

    if internal_npixels % sample_fac_os != 0:
        raise ValueError(
            f"Internal grid of {internal_npixels} pixels cannot be "
            f"downsampled by {sample_fac_os}."
        )

    final_npixels = internal_npixels // sample_fac_os
    final_pixel_scale = (
        psf_pixel_scale / oversample * sample_fac_os
    )

    print(
        f"sample_fac: {sample_fac}\n"
        f"nominal grid: {psf_npixels} pixels\n"
        f"internal grid: {internal_npixels} pixels\n"
        f"downsample factor: {sample_fac_os}\n"
        f"final grid: {final_npixels} pixels\n"
        f"final scale: {final_pixel_scale} arcsec/pixel"
    )

    optics = dl.AngularOpticalSystem(
        wf_npixels=wf_npixels,
        diameter=diameter,
        layers=layers,
        psf_npixels=psf_npixels,
        psf_pixel_scale=psf_pixel_scale,
        oversample=oversample
    )

    wlmin_data = 1.0e-6
    wlmax_data = 2.0e-6

    wavelengths_data = np.linspace(wlmin_data, wlmax_data, n_wavels_data)

    nbasis = n_wavels_data
    bandpass = np.ones(n_wavels_data)

    spectrum_bins = CombinedBasisSpectrum(
        wavelengths_data,
        bandpass,
        np.ones(nbasis),
        np.eye(nbasis)
    )

    source_data = dl.PointSource(
        spectrum=spectrum_bins
    )

    # detector downsampling
    detector = dl.LayeredDetector([
        (
            'downsample',
            dl.layers.detector_layers.Downsample(sample_fac_os)
        )
    ])

    model_data = dl.Telescope(optics, source_data, detector)

    def model_psf(spectrum, instrument):
        old_spec = instrument.get("source.spectrum")

        new_spec = CombinedBasisSpectrum(
            wavels=old_spec.wavelengths,
            filt_weights=old_spec.filt_weights,
            basis_weights=spectrum,
            basis=old_spec.basis_vects,
        )

        telescope = instrument.set("source.spectrum", new_spec)
        telescope = telescope.set("source.flux", telescope.get("source.spectrum").flux)
        return telescope.model()

    injected_spectrum = np.ones(n_wavels_data) / n_wavels_data * flux

    img_full = model_psf(injected_spectrum, model_data)

    def loss_fn(params, model, data):
        spectrum = params['spectrum']
        img = model_psf(spectrum, model)
        loss = -poisson_loglike_gamma(data, img).sum()
        # loss = -normal_logpdf(data, img, np.sqrt(data)).sum()
        return loss

    params = {
        "spectrum": injected_spectrum
    }

    f = lambda params: loss_fn(params, model_data, img_full)

    H, unflatten = hessian(f, params)

    H = np.asarray(H)
    H = 0.5 * (H + H.T)

    eigvals, eigvecs = np.linalg.eigh(H)

    idx = np.argsort(eigvals)[::-1]

    eigvals = eigvals[idx]
    eigvecs = eigvecs[:, idx]

    # Optional sign convention
    for i in range(eigvecs.shape[0]):
        eigvecs = eigvecs.at[:, i].mul(np.sign(np.mean(eigvecs[:1, i])))

    a = eigvecs.T @ injected_spectrum

    relstd = 1 / (np.abs(a[0]) * np.sqrt(eigvals))

    # safer than np.argmax(relstd > threshold)
    # because np.argmax returns 0 if nothing is True
    bad = np.where(relstd > threshold)[0]

    if len(bad) == 0:
        nconstrained_modes = len(relstd)
    else:
        nconstrained_modes = bad[0]

    return nconstrained_modes

In [ ]:


for i, fov in enumerate(fovs):
    for j, sample_fac in enumerate(sample_facs):
        print(f"fov={fov:.2f}, read_var={read_var:.2f}")

        nconstrained_sample = nconstrained_sample.at[i, j].set(calculate_nconstrained_modes_sample(
            fov=fov,
            flux=flux,
            sample_fac=sample_fac,
            threshold=0.1
        ))

        print(nconstrained_sample[i, j])

In [ ]:
for i in range(len(fovs)):
    plt.plot(sample_facs,nconstrained_sample[i,:])

In [ ]:
# initialise an fov (perhaps 8) with a fixed number of pixels (perhaps enough to doubly sample). then just increase the pixel scale and see what happen.
# or i can just do an array of fovs with this trick and that pops out.
# pretty sweet idea

In [ ]:
# Results of the two grid searches above (cells 4 and 7), so the figure below can be
# remade without re-running them. Re-save if either search changes:
# np.save('read_noise_grid.npy', nconstrained_grid)
# np.save('read_noise_sample.npy', nconstrained_sample)
nconstrained_grid = np.load('read_noise_grid.npy')
nconstrained_sample = np.load('read_noise_sample.npy')


In [ ]:
# import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe
path_effects = [pe.Stroke(linewidth=1.5, foreground='k'), pe.Normal()]
path_effects_2 = [pe.Stroke(linewidth=2.5, foreground='k'), pe.Normal()]

colors = plt.colormaps["cividis"](
    np.linspace(0.0, 1.0, 5)
)

markers = [
    "s",
    "o",
    "x",
    "+",
    "_"
]

markersizes = [
    3,
    3,
    3,
    5,
    5,
]

labels = [
    r'$4\ \lambda_{\min}/D$',
    r'$8\ \lambda_{\min}/D$',
    r'$16\ \lambda_{\min}/D$',
    r'$32\ \lambda_{\min}/D$',
    r'$64\ \lambda_{\min}/D$',
]

linestyles = [
    '-.',
    (0, (3, 1, 1, 1, 1, 1)),
    '-',
    '--',
    ':',
]

fig, (ax1, ax2) = plt.subplots(
    1,
    2,
    figsize=(fig_width, 0.45 * fig_width),
    sharey=True,
    gridspec_kw={
        "width_ratios": [0.5, 0.5],
        # "wspace": 0.06,
    },
    constrained_layout=True,
)

# LEFT PANEL: FOVS
for i in range(len(fovs)):
    ax1.plot(read_vars, nconstrained_grid[i, :],
             color=colors[i],
             path_effects=path_effects,
             solid_capstyle='round',
             marker=markers[i],
             markerfacecolor='none',
             markeredgewidth=1.0,
             markersize=markersizes[i],
             label=labels[i],
             ls=linestyles[i],
             lw=1.0,)

ax1.set_xticks([0, 16, 32, 48, 64])
ax1.set_ylim(0,31)

secax1 = ax1.secondary_yaxis('right',
    functions=(lambda N: 1.5*N, lambda R: R/1.5))
secax2 = ax2.secondary_yaxis('right',
    functions=(lambda N: 1.5*N, lambda R: R/1.5))

secax1.set_yticklabels([])      # hide numbers
secax1.tick_params(direction='in')

secax2.tick_params(direction='in')
secax2.set_ylabel(r'$R$')

ax1.tick_params(
    axis='both',
    direction='in',
    top=True,
    # right=True
)
ax1.set_ylabel(r'$N$')
ax1.legend(handlelength=1.0, frameon=False, markerfirst=False)

js.panel_label(ax1, "a", x=0.08, y=0.84, va="bottom")

for i in range(len(fovs)):
    ax2.plot(sample_facs, nconstrained_sample[i,:],
            color=colors[i],
            path_effects=path_effects,
            solid_capstyle='round',
            marker=markers[i],
            markerfacecolor='none',
            markeredgewidth=1.0,
            markersize=markersizes[i],
            # label=labels[i],
            ls=linestyles[i],
            lw=1.0)
    
ax1.set_xlabel(r'Read noise density $\big[\mathrm{e}^{-} \big]$')

logx = np.log2(sample_facs)

nfov, nsamp = nconstrained_sample.shape

for k in range(-nfov + 1, nsamp):
    i = np.arange(nfov)
    j = i + k
    m = (j >= 0) & (j < nsamp)

    i = i[m]
    j = j[m]

    if len(i) < 2:
        continue

    x = sample_facs[j]
    y = nconstrained_sample[i, j]

    # smooth-ish guide in log-x space
    xx = np.linspace(np.log2(x[0]), np.log2(x[-1]), 200)
    yy = np.interp(xx, np.log2(x), y)

    ax2.plot(
        2**xx,
        yy,
        color='0.5',
        alpha=0.25,
        lw=1.5,
        zorder=0,
        solid_capstyle='round',
    )

ax2.set_xscale('log', base=2)
ax2.tick_params(
    axis='both',
    direction='in',
    top=True,
    # right=True
)
# ax2.legend(handlelength=2.0, frameon=False)

ax2.plot(
    [], [],
    color='0.5',
    alpha=0.5,
    lw=1.5,
    label=r'$N_{\rm pix}=\mathrm{const.}$',
)

ax2.legend(frameon=False, handlelength=1.0, markerfirst=False, handletextpad=0.4)

js.panel_label(ax2, "b", x=0.08, y=0.84, va="bottom")

ax2.set_xlabel(r'Undersampling factor')

plt.savefig('read_nyquist_jatis.pdf', dpi=300, backend='pgf')
plt.show()